In [1]:
from langchain_ollama import OllamaLLM
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.retrievers import EnsembleRetriever
from langchain.chains import RetrievalQA

import os

- RAG retrievers (via ChromaDB library)

In [2]:
import chromadb

# Initialize the persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path="../output/vectorstore")


# List all collections in the database
collections = chroma_client.list_collections()
print("Collections in the database:")
for collection in collections:
    # Print the collection name directly since collection is now a string
    print(collection)

# Delete a specific collection if needed
# chroma_client.delete_collection(name="base_address")

# If you need to work with a specific collection, use get_collection():
address = chroma_client.get_collection(name="base_address")

# show the number of documents in the collection
# print(address.count())



Collections in the database:
Collection(name=base_address)


- Similarity Search (via ChromaDB library)

In [3]:
result = \
address.query(
    query_texts=[("BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN").upper()], # Chroma will embed this for you
    n_results=1 # how many results to return
)
result

InvalidArgumentError: Error executing plan: Error sending backfill request to compactor: Failed to pull logs from the log store

In [ ]:
result['metadatas'][0][0]['district']

In [ ]:
result['metadatas'][0][0]['state']

In [ ]:
print(f"Consine Similarity Score: {result['distances'][0][0]}") # lower is better
print(f"{result['documents'][0][0]}")

- RAG retrivers (via Langchain library)

In [ ]:

from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", cache_dir="../local_model")
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", cache_dir="../local_model")


In [4]:

# Initialize the embedding
# oembed = OllamaEmbeddings(base_url="http://localhost:11434", model="llama3.2:latest") # 3072-dim
# hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim ~ pull from online
hfembed = HuggingFaceEmbeddings(model_name="../local_model/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf")

# Connect to existing vectorstore
vectorstore = Chroma(
    collection_name="base_address",
    embedding_function=hfembed,
    persist_directory="../output/vectorstore"
)

C:\Users\izard\AppData\Local\Temp\ipykernel_19284\2333871321.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hfembed = HuggingFaceEmbeddings(model_name="../local_model/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf")
c:\Users\izard\miniconda3\envs\etl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name ../local_model/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9

In [5]:
query = "BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN"

# Generate embeddings for the query (depends on your embedding setup)
query_embedding = hfembed.embed_query(query.upper())

# Perform the query using the generated embedding
results = address.query(
    query_embeddings=[query_embedding],
    n_results=5  # adjust based on how many matches you want
)

# Show the result , lower cosine similarity score is better
for distance, doc in zip(results['distances'][0], results['documents'][0]):
    print(f"{distance:.4f} → {doc}")



InvalidArgumentError: Error executing plan: Error sending backfill request to compactor: Failed to pull logs from the log store

- Calculation using SkLearn

In [ ]:
query1 = "BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN"
# Initialize the embedding
embed1=hfembed.embed_query((query1.upper()))

query2 = "AMPANGAN, 70400, SEREMBAN, NEGERI SEMBILAN"
embed2=hfembed.embed_query(query2)

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim_score = cosine_similarity(
    np.array(embed1).reshape(1, -1),
    np.array(embed2).reshape(1, -1)
)

print("Cosine similarity:", sim_score[0][0])


In [ ]:
vectorstore.search("BANDAR WARISAN PUTERI JENIS AMETHYST,AMPANGAN,SEREMBAN,NEGERI SEMBILAN",k=5,search_type="similarity")

In [ ]:
results = vectorstore.similarity_search_with_score(str.upper("PERSIARAN MAYANG PASIR, BAYAN LEPAS, PULAU PINANG"), k=5)
for doc, score in results:
    print(f"{score:.4f} → {doc.page_content}")


- Combine RAG Result & LLM

In [ ]:
# Initialize the LLM
llm = OllamaLLM(model="llama3.2:latest", base_url="http://localhost:11434")

In [ ]:
# Test LLM
llm.invoke("Hello world")

In [ ]:
address.query(
    query_embeddings=[query_embedding],
    n_results=5  # adjust based on how many matches you want
)


In [ ]:
import re
import json


postcode_matcher="PERSIARAN MAYANG PASIR"

# Generate embeddings for the query (depends on your embedding setup)
query_embedding = hfembed.embed_query(postcode_matcher.upper())

# Perform the query using the generated embedding
results = address.query(
    query_embeddings=[query_embedding],
    n_results=5  # adjust based on how many matches you want
)

max_attempts = 3
attempt = 0
postcode = None

while attempt < max_attempts and postcode is None:
    answer = llm.invoke(
        f"Given the following informations {results['documents'][0][0]} with similarity score {results['distances'][0][0]} ; \
        {results['documents'][0][1]} with similarity score {results['distances'][0][1]} ; \
        {results['documents'][0][2]} with similarity score {results['distances'][0][2]} ; \
        {results['documents'][0][3]} with similarity score {results['distances'][0][3]} ; \
        {results['documents'][0][4]} with similarity score {results['distances'][0][4]} for evaluation. \
        What is the possible postcode and the similarity_score for {postcode_matcher} strictly based on the informations?\
        Answer 1 postcode and score value only in json format"
    )
    json_match = re.search(r"\{[\s\S]*?\}", answer)
    if json_match:
        extracted_json = json_match.group(0)
        postcode = json.loads(extracted_json)["postcode"]
        similarity_distance = json.loads(extracted_json)["similarity_score"]
        print(postcode)
        print(similarity_distance)
    else:
        print("No JSON found in answer. Retrying...")
        attempt += 1

if postcode is None:
    print("Failed to extract JSON after multiple attempts.")


In [ ]:
import sys
sys.path.append("..")  # or the actual path to the script
from postcode import malaysia_postcode

print(malaysia_postcode("GRN237447 LOT11499 SEKSYEN 1 (HSD16524,PT1560), TAMAN KASIH PUTERA,PEKAN BAHAU, JEMPOL, NEGERI SEMBILAN,PEKAN BAHAU,JEMPOL"))